# Tugas 1 — Klasifikasi Wine Quality dengan KNN

| Nama | NRP |
|---|---|
| Prima Surya Nusantara | 5054251036 |

## Deskripsi Tugas
Pada tugas ini, kita akan menggunakan Wine Quality dfset. dfset bisa diakses melalui link berikut:\
🔗 https://www.kaggle.com/dfsets/yasserh/wine-quality-dfset

Tujuan utama dari tugas ini adalah membangun model K-Nearest Neighbors (KNN) untuk mengklasifikasikan kualitas wine. 

Langkah-langkah yang harus dilakukan antara lain:
1. Persiapan dfset & Eksplorasi Awal

- Memuat dfset, melihat struktur df, dan distribusi label.

2. Preprocessing 
- Memproses df agar siap untuk digunakan dalam membangun model.

3. Eksperimen Model KNN
- Bangun model KNN dengan mencoba beberapa nilai k (misalnya 3, 5, dan 7 --> BEBAS) serta dua metric jarak (seperti Euclidean dan Manhattan).
- Eksperimen ini bertujuan untuk membandingkan performa KNN dengan parameter yang berbeda.

4. Evaluasi Model
- Hitung metrik evaluasi seperti Accuracy, Precision, Recall, F1-Score, serta visualisasikan Confusion Matrix.

5. Analisis & Kesimpulan
- Bandingkan hasil antar eksperimen yang telah dilakukan dan berikan kesimpulan.

## 1. Persiapan dataset dan eksplorasi awal

Sumber berkas: `WineQT.csv`, input **Wine Quality Dataset** yang digunakan pada notebook awal.
Nama dan identitas mahasiswa dipertahankan. Unit kimia tidak diasumsikan karena berkas CSV tidak menyertakan kamus satuan.
Semua angka deskriptif dihitung langsung dari berkas yang dibaca.

In [1]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from IPython.display import display, Markdown
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
SEED = 42
LABELS = [3, 4, 5, 6, 7, 8]
K_VALUES = [1, 3, 5, 7, 9]
def tabel_format(data, format_kolom):
    # Salinan tampilan saja; nilai numerik asli tetap tersedia untuk analisis.
    tampilan = data.copy()
    for kolom, pola in format_kolom.items():
        tampilan[kolom] = tampilan[kolom].map(pola.format)
    return tampilan

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12, 'axes.labelsize': 10})
print(f'Python {sys.version.split()[0]} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}')

Python 3.12.13 | pandas 2.3.3 | scikit-learn 1.6.1


In [2]:
path_data = Path('/kaggle/input/datasets/yasserh/wine-quality-dataset/WineQT.csv')
df_raw = pd.read_csv(path_data)
print(f'Data berhasil dimuat: {len(df_raw):,} baris dan {df_raw.shape[1]} kolom.')
print('SHA-256 berkas:', hashlib.sha256(path_data.read_bytes()).hexdigest())
display(Markdown('### Contoh lima observasi pertama'))
display(df_raw.head())
display(Markdown('### Struktur dan kelengkapan data'))
audit = pd.DataFrame({
    'Tipe data': df_raw.dtypes.astype(str),
    'Nilai kosong': df_raw.isna().sum(),
    'Nilai unik': df_raw.nunique()
})
display(audit)

Data berhasil dimuat: 1,143 baris dan 13 kolom.
SHA-256 berkas: 7e38cc28812d08f521ee19e29e9d3622cde03464ff5e9a8b14aa991ec74ae49e


NameError: name 'Markdown' is not defined

### Cara memahami data

- `quality` adalah label yang ingin diprediksi, bukan fitur input.
- `Id` adalah identitas, bukan karakteristik wine.
- Kolom lain merupakan 11 fitur numerik. Encoding kategori tidak diperlukan.
- Nilai kosong perlu dibedakan dari angka nol. Nol tidak otomatis berarti data hilang.
- Label kualitas bersifat berurutan, tetapi KNN classifier di sini memperlakukannya sebagai kelas terpisah.

## 2. Preprocessing dan rancangan evaluasi

### 2.1 Missing value dan duplikasi

In [ ]:
n_nan = int(df_raw.isna().sum().sum())
n_dup_id = int(df_raw.duplicated().sum())
data_no_id = df_raw.drop(columns='Id')
n_dup = int(data_no_id.duplicated().sum())
if n_nan:
    raise ValueError('Dataset berubah dan memiliki nilai kosong. Tambahkan imputasi yang di-fit pada training saja.')
assert all(pd.api.types.is_numeric_dtype(t) for t in data_no_id.dtypes)
assert np.isfinite(data_no_id.to_numpy()).all(), 'Ada nilai tak hingga.'
df = data_no_id.drop_duplicates().reset_index(drop=True)
display(pd.DataFrame({
    'Pemeriksaan': ['Baris awal', 'Nilai kosong', 'Duplikat termasuk Id',
                    'Duplikat tanpa Id (kelebihan baris)', 'Baris setelah deduplikasi'],
    'Jumlah': [len(df_raw), n_nan, n_dup_id, n_dup, len(df)]
}))
print('Tidak ada nilai kosong: tidak dilakukan penghapusan NaN maupun imputasi.')
X = df.drop(columns='quality')
y = df['quality'].astype(int)
print(f'Fitur X: {X.shape[0]:,} observasi × {X.shape[1]} fitur; target y: {len(y):,} label.')

### 2.2 Split satu kali: 80% training, 20% testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_test) == len(X)
distribusi = pd.DataFrame({
    'Seluruh data': y.value_counts(),
    'Training': y_train.value_counts(),
    'Testing': y_test.value_counts()
}).reindex(LABELS, fill_value=0).fillna(0).astype(int)
distribusi.index.name = 'Kualitas'
distribusi['Proporsi seluruh data (%)'] = (100 * distribusi['Seluruh data'] / len(y)).round(2)
display(distribusi)
print(f'Training: {len(X_train)} baris | Testing: {len(X_test)} baris')
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(distribusi.index.astype(str), distribusi['Training'], color='#267c8e')
ax.bar_label(bars, padding=3)
ax.set(title='Kelas kualitas 5 dan 6 mendominasi data training',
       xlabel='Kelas kualitas wine', ylabel='Jumlah observasi training')
ax.set_ylim(0, distribusi['Training'].max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
data_eda = X_train.assign(quality=y_train)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(data_eda.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8})
ax.set_title('Korelasi Pearson pada data training')
plt.tight_layout()
plt.show()
korelasi_target = data_eda.corr()['quality'].drop('quality').sort_values()
display(korelasi_target.rename('Korelasi terhadap quality').to_frame().round(3))
display(Markdown(
    f'Hubungan linear positif terbesar dengan quality terdapat pada **{korelasi_target.idxmax()}** '
    f'({korelasi_target.max():.3f}); hubungan negatif terbesar pada **{korelasi_target.idxmin()}** '
    f'({korelasi_target.min():.3f}). Korelasi tidak membuktikan sebab-akibat atau pentingnya fitur dalam KNN.'
))

### 2.3 Memeriksa outlier pada data training

Kotak boxplot memuat 50% data tengah (Q1–Q3), garis di dalamnya adalah median.
Whisker mencapai observasi terjauh yang masih berada dalam pagar **Q1 − 1,5 × IQR** dan **Q3 + 1,5 × IQR**.
Titik di luar whisker adalah calon outlier, bukan otomatis pengukuran salah.
Setiap fitur ditampilkan dengan sumbu sendiri agar fitur berskala kecil tetap terbaca.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 12))
for ax, kolom in zip(axes.flat, X_train.columns):
    sns.boxplot(y=X_train[kolom], ax=ax, color='#77b8bd', fliersize=3)
    ax.set(title=kolom, xlabel='', ylabel='Nilai fitur')
for ax in list(axes.flat)[X_train.shape[1]:]:
    ax.axis('off')
fig.suptitle('Calon outlier per fitur — data training', y=1.01)
plt.tight_layout()
plt.show()

def mask_inlier(data):
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    calon_outlier = data.lt(q1 - 1.5 * iqr) | data.gt(q3 + 1.5 * iqr)
    return ~calon_outlier.any(axis=1), calon_outlier

keep_train, outlier_train = mask_inlier(X_train)
ringkasan_outlier = pd.DataFrame({
    'Jumlah calon outlier': outlier_train.sum(),
    'Persentase training (%)': (100 * outlier_train.mean()).round(2)
}).sort_values('Jumlah calon outlier', ascending=False)
display(ringkasan_outlier)
retensi = pd.DataFrame({
    'Training awal': y_train.value_counts(),
    'Training lolos IQR': y_train.loc[keep_train].value_counts()
}).reindex(LABELS).fillna(0).astype(int)
retensi['Dihapus'] = retensi['Training awal'] - retensi['Training lolos IQR']
retensi.index.name = 'Kualitas'
display(retensi)

### 2.4 Dua skenario dengan scaler yang sama

1. **Pertahankan outlier:** gunakan seluruh baris training.
2. **Hapus outlier training:** hapus baris training yang melewati pagar IQR pada minimal satu fitur.

## 3. Eksperimen model KNN

Uji **20 konfigurasi**: 2 skenario outlier × 2 jarak × 5 nilai k.

- **Euclidean:** jarak garis lurus berdasarkan akar jumlah kuadrat selisih fitur.
- **Manhattan:** jumlah nilai absolut selisih fitur.
- **k kecil:** lebih sensitif pada observasi lokal/noise; k=1 biasanya memiliki akurasi training sangat tinggi.
- **k besar:** keputusan lebih halus, tetapi dapat mengabaikan kelas langka.

Gunakan **StratifiedKFold 3-fold** pada training untuk pemilihan konfigurasi. Tiga fold dipilih karena
kelas terlangka berisi sedikit sampel. F1-macro tetap dihitung untuk seluruh kelas 3–8.
Urutan pemilihan: F1-macro CV tertinggi, lalu accuracy CV tertinggi jika seri.
Test tidak dipakai untuk memilih k, metrik jarak, atau skenario outlier.

In [ ]:
def latih_knn(X_fit, y_fit, hapus_outlier, k, jarak):
    keep = mask_inlier(X_fit)[0] if hapus_outlier else pd.Series(True, index=X_fit.index)
    X_pakai, y_pakai = X_fit.loc[keep], y_fit.loc[keep]
    if len(X_pakai) < k:
        raise ValueError('Jumlah data training setelah IQR kurang dari k.')
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k, metric=jarak, weights='uniform')
    )
    model.fit(X_pakai, y_pakai)
    return model, keep

def f1_macro(y_asli, y_prediksi):
    return f1_score(y_asli, y_prediksi, labels=LABELS, average='macro', zero_division=0)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
folds = list(cv.split(X_train, y_train))
hasil_cv = []
for hapus in [False, True]:
    for jarak in ['euclidean', 'manhattan']:
        for k in K_VALUES:
            skor_f1, skor_acc, skor_train, n_fit = [], [], [], []
            for idx_fit, idx_val in folds:
                X_fit, X_val = X_train.iloc[idx_fit], X_train.iloc[idx_val]
                y_fit, y_val = y_train.iloc[idx_fit], y_train.iloc[idx_val]
                model, keep = latih_knn(X_fit, y_fit, hapus, k, jarak)
                pred_val = model.predict(X_val)
                skor_f1.append(f1_macro(y_val, pred_val))
                skor_acc.append(accuracy_score(y_val, pred_val))
                skor_train.append(accuracy_score(y_fit, model.predict(X_fit)))
                n_fit.append(int(keep.sum()))
            hasil_cv.append({
                'Skenario': 'Hapus outlier training' if hapus else 'Pertahankan outlier',
                'Hapus outlier': hapus, 'Jarak': jarak, 'k': k,
                'F1-macro CV': np.mean(skor_f1), 'SD F1 CV': np.std(skor_f1, ddof=1),
                'Accuracy CV': np.mean(skor_acc),
                'Accuracy train CV': np.mean(skor_train), 'Rata-rata n fit': np.mean(n_fit)
            })
hasil_cv = pd.DataFrame(hasil_cv).sort_values(
    ['F1-macro CV', 'Accuracy CV'], ascending=False, kind='stable'
).reset_index(drop=True)
best = hasil_cv.iloc[0].copy()  # Dikunci sebelum test dihitung.
kolom_cv = ['Skenario', 'Jarak', 'k', 'F1-macro CV', 'SD F1 CV', 'Accuracy CV', 'Accuracy train CV']
display(tabel_format(hasil_cv[kolom_cv], {c: '{:.2%}' for c in kolom_cv if 'CV' in c}))
print(f"Konfigurasi terpilih dari CV: {best['Skenario']}, k={best['k']}, jarak {best['Jarak']}.")
print(f"F1-macro CV = {best['F1-macro CV']:.2%}; SD antar-fold = {best['SD F1 CV']:.2%}.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, skenario in zip(axes, ['Pertahankan outlier', 'Hapus outlier training']):
    for jarak, warna in [('euclidean', '#267c8e'), ('manhattan', '#c66b30')]:
        bagian = hasil_cv[(hasil_cv['Skenario'] == skenario) & (hasil_cv['Jarak'] == jarak)].sort_values('k')
        ax.errorbar(bagian['k'], 100 * bagian['F1-macro CV'],
                    yerr=100 * bagian['SD F1 CV'], marker='o', capsize=3, color=warna, label=jarak.title())
    ax.set(title=skenario, xlabel='Jumlah tetangga (k)', xticks=K_VALUES, ylim=(0, 100))
    ax.legend(title='Metrik jarak')
axes[0].set_ylabel('F1-macro validasi silang (%)')
fig.suptitle('Pemilihan model berdasarkan validasi, bukan test', y=1.03)
plt.tight_layout()
plt.show()
print('Error bar = ±1 standar deviasi dari 3 fold, bukan confidence interval.')

### 3.1 Rekap accuracy train dan test


In [ ]:
rekap = []
models = {}
for _, row in hasil_cv.iterrows():
    hapus, k, jarak = bool(row['Hapus outlier']), int(row['k']), row['Jarak']
    model, keep = latih_knn(X_train, y_train, hapus, k, jarak)
    models[(hapus, k, jarak)] = model
    pred_train, pred_test = model.predict(X_train), model.predict(X_test)
    rekap.append({
        'Skenario': row['Skenario'], 'Jarak': jarak, 'k': k, 'Baris fit': int(keep.sum()),
        'Accuracy train': accuracy_score(y_train, pred_train),
        'Accuracy test': accuracy_score(y_test, pred_test),
        'F1-macro test': f1_macro(y_test, pred_test), 'F1-macro CV': row['F1-macro CV']
    })
hasil_eksperimen = pd.DataFrame(rekap)
display(tabel_format(hasil_eksperimen, {c: '{:.2%}' for c in
    ['Accuracy train', 'Accuracy test', 'F1-macro test', 'F1-macro CV']}))
knn_terbaik = models[(bool(best['Hapus outlier']), int(best['k']), best['Jarak'])]
y_pred = knn_terbaik.predict(X_test)
assert len(y_pred) == len(y_test)

## 4. Evaluasi model terpilih

In [ ]:
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)
def metrik_lengkap(y_true, pred):
    return {
        'Accuracy': accuracy_score(y_true, pred),
        'Precision macro': precision_score(y_true, pred, labels=LABELS, average='macro', zero_division=0),
        'Recall macro': recall_score(y_true, pred, labels=LABELS, average='macro', zero_division=0),
        'F1-macro': f1_macro(y_true, pred),
        'F1-weighted': f1_score(y_true, pred, labels=LABELS, average='weighted', zero_division=0),
        'Balanced accuracy': balanced_accuracy_score(y_true, pred)
    }
skor_final = metrik_lengkap(y_test, y_pred)
skor_baseline = metrik_lengkap(y_test, pred_baseline)
evaluasi = pd.DataFrame({'KNN terpilih': skor_final, 'Baseline kelas mayoritas': skor_baseline})
display(tabel_format(evaluasi, {c: '{:.2%}' for c in evaluasi.columns}))
print(f'Prediksi benar: {(y_pred == y_test.to_numpy()).sum()} dari {len(y_test)} observasi test.')
laporan = classification_report(y_test, y_pred, labels=LABELS, output_dict=True, zero_division=0)
per_kelas = pd.DataFrame(laporan).T.loc[[str(k) for k in LABELS], ['precision', 'recall', 'f1-score', 'support']]
per_kelas.index.name = 'Kelas quality'
per_kelas['support'] = per_kelas['support'].astype(int)
display(tabel_format(per_kelas, {'precision': '{:.2%}', 'recall': '{:.2%}', 'f1-score': '{:.2%}', 'support': '{:d}'}))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=LABELS)
cm_normal = np.divide(cm, cm.sum(axis=1, keepdims=True),
                      out=np.zeros_like(cm, dtype=float), where=cm.sum(axis=1, keepdims=True) != 0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS,
            ax=axes[0], cbar=False)
sns.heatmap(cm_normal, annot=True, fmt='.0%', cmap='Blues', vmin=0, vmax=1,
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[1], cbar=False)
for ax, judul in zip(axes, ['Jumlah observasi', 'Persentase per kelas aktual']):
    ax.set(title=judul, xlabel='Kelas prediksi', ylabel='Kelas aktual')
fig.suptitle(f"Confusion matrix — KNN k={int(best['k'])}, {best['Jarak']}", y=1.02)
plt.tight_layout()
plt.show()
print('Diagonal = prediksi benar. Di luar diagonal = kelas yang tertukar. Grafik kanan berjumlah 100% per baris.')

## 5. Analisis hasil eksperimen

In [ ]:
terbaik_skenario = hasil_cv.sort_values(['F1-macro CV', 'Accuracy CV'], ascending=False).groupby('Skenario', sort=False).head(1)
display(tabel_format(terbaik_skenario[['Skenario', 'Jarak', 'k', 'F1-macro CV', 'SD F1 CV']],
    {'F1-macro CV': '{:.2%}', 'SD F1 CV': '{:.2%}'}))
terbaik_jarak = hasil_cv.sort_values(['F1-macro CV', 'Accuracy CV'], ascending=False).groupby('Jarak', sort=False).head(1)
display(tabel_format(terbaik_jarak[['Jarak', 'Skenario', 'k', 'F1-macro CV']], {'F1-macro CV': '{:.2%}'}))

paired = hasil_cv.pivot(index=['Jarak', 'k'], columns='Skenario', values='F1-macro CV')
delta_cv = paired['Hapus outlier training'] - paired['Pertahankan outlier']
display((100 * delta_cv).rename('Selisih F1-macro CV: hapus − pertahankan (poin persen)').to_frame().round(2))
kelas_gagal = ', '.join(per_kelas.index[per_kelas['recall'] == 0].tolist()) or 'tidak ada'
cm_salah = cm.copy()
np.fill_diagonal(cm_salah, 0)
i, j = np.unravel_index(cm_salah.argmax(), cm_salah.shape)
baris_best = hasil_eksperimen[
    (hasil_eksperimen['Skenario'] == best['Skenario']) &
    (hasil_eksperimen['Jarak'] == best['Jarak']) & (hasil_eksperimen['k'] == best['k'])
].iloc[0]
pilihan_k1 = hasil_eksperimen[(hasil_eksperimen['k'] == 1) &
                             (hasil_eksperimen['Skenario'] == 'Pertahankan outlier')]
narasi_analisis = f'''### 5.1 Pemilihan konfigurasi

Konfigurasi terpilih adalah **{best['Skenario']}, k={int(best['k'])}, jarak {best['Jarak']}**.
F1-macro validasi silang rata-rata **{best['F1-macro CV']:.2%}**.

### 5.2 Pengaruh outlier

Aturan IQR menandai **{int((~keep_train).sum())} dari {len(X_train)} baris training**
({(~keep_train).mean():.2%}). Skenario penghapusan menggunakan {int(keep_train.sum())} baris untuk fit akhir,
sedangkan skenario mempertahankan menggunakan {len(X_train)} baris. Keduanya dinilai pada **{len(X_test)} observasi test yang sama**.
Pada {int((delta_cv > 0).sum())} dari {len(delta_cv)} pasangan k–jarak, penghapusan menghasilkan F1-macro CV lebih tinggi.
Rata-rata selisih berpasangan adalah **{100 * delta_cv.mean():+.2f} poin persentase**.

### 5.3 Generalisasi dan k

Untuk k=1 tanpa penghapusan, accuracy training berada pada
{pilihan_k1['Accuracy train'].min():.2%}, sedangkan accuracy test {pilihan_k1['Accuracy test'].min():.2%}.
Kesenjangan ini konsisten dengan overfitting; KNN k=1 memakai observasi training itu sendiri sebagai tetangga.
Untuk model terpilih, accuracy pada seluruh training adalah {baris_best['Accuracy train']:.2%}
dan pada test {skor_final['Accuracy']:.2%}.

### 5.4 Kinerja per kelas dan baseline

Model terpilih memperoleh **accuracy test {skor_final['Accuracy']:.2%}**, **F1-macro {skor_final['F1-macro']:.2%}**,
dan **F1-weighted {skor_final['F1-weighted']:.2%}**. Baseline yang selalu menebak kelas mayoritas
memperoleh accuracy {skor_baseline['Accuracy']:.2%} dan F1-macro {skor_baseline['F1-macro']:.2%}.
Selisih accuracy KNN terhadap baseline adalah {100 * (skor_final['Accuracy'] - skor_baseline['Accuracy']):+.2f} poin persentase.
Kelas dengan recall nol: **{kelas_gagal}**. Recall nol berarti tidak ada anggota aktual kelas itu yang diprediksi benar,
bukan selalu berarti model tidak pernah memprediksi kelas tersebut.
Kesalahan paling sering: kualitas aktual **{LABELS[i]}** diprediksi sebagai **{LABELS[j]}**, sebanyak **{cm_salah[i, j]}** observasi.
Perbedaan F1-macro dan F1-weighted menunjukkan pentingnya memeriksa performa kelas minoritas secara terpisah.
'''
display(Markdown(narasi_analisis))

## 6. Kesimpulan dan saran

In [ ]:
narasi_penutup = f'''### Kesimpulan

1. Dataset awal berisi {len(df_raw):,} observasi dan {df_raw.shape[1]} kolom, tanpa nilai kosong.
   Setelah Id dikeluarkan dan {n_dup} kelebihan baris identik dideduplikasi sesuai asumsi eksperimen,
   terdapat {len(df):,} observasi dengan 11 fitur dan satu target quality.
2. Kelas kualitas tidak seimbang. Evaluasi perlu memadukan accuracy, F1-macro, dan confusion matrix;
   accuracy saja tidak cukup menjelaskan kemampuan mengenali kelas langka.
3. Dari 20 konfigurasi yang diuji, **{best['Skenario']}, k={int(best['k'])}, {best['Jarak']}** terpilih
   berdasarkan F1-macro validasi silang. Pada test yang sama, model memperoleh accuracy **{skor_final['Accuracy']:.2%}**,
   F1-macro **{skor_final['F1-macro']:.2%}**, dan balanced accuracy **{skor_final['Balanced accuracy']:.2%}**.
4. Penghapusan outlier bukan langkah yang wajib atau selalu menguntungkan. Hasil perlu dinilai dengan data uji yang sama,
   scaler yang dikontrol, dan pemeriksaan dampak terhadap kelas minoritas.
5. Model ini merupakan baseline pembelajaran, belum cukup untuk dinyatakan andal bagi seluruh tingkat kualitas wine.

### Saran

1. **Prioritaskan data kelas langka.** Tambahkan observasi kualitas rendah/tinggi dan verifikasi identitas sampel serta nilai ekstrem.
2. **Lanjutkan tuning melalui validasi, bukan test.** Uji weights='distance', rentang k tambahan, dan StandardScaler versus
   RobustScaler menggunakan fold yang sama. Pilih konfigurasi berdasarkan F1-macro beserta recall per kelas.
3. **Tangani outlier secara hati-hati.** Bandingkan mempertahankan, membatasi nilai, atau transformasi fitur;
   pelajari seluruh parameter hanya dari training. Jangan membersihkan test untuk menaikkan skor.
4. **Bandingkan dengan baseline model lain.** Misalnya Logistic Regression atau Random Forest dengan konfigurasi
   penyeimbangan kelas yang sesuai. KNN tidak memiliki parameter class_weight.
5. **Perkuat validasi.** Gunakan repeated/nested stratified cross-validation dengan jumlah fold sesuai kelas terkecil,
   lalu dataset eksternal untuk konfirmasi. Jika label ingin digabung, tetapkan alasan substantif lebih dahulu karena itu
   mengubah pertanyaan penelitian, bukan sekadar memperbaiki skor.
'''
display(Markdown(narasi_penutup))